# Data Preparation

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from google.colab import files

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# 1. Upload dataset from local machine to Google Colab
print("Please upload your dataset CSV file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load dataset into our descriptive pandas DataFrame
product_catalog_df = pd.read_csv(filename)
print(f"\n[INFO] Original dataset shape: {product_catalog_df.shape}")

# 2. Drop unnecessary columns
# Columns like 'Unnamed: 0' (index artifact), 'image', and 'link' are irrelevant for semantic similarity search.
columns_to_drop = ['Unnamed: 0', 'image', 'link']
product_catalog_df = product_catalog_df.drop(columns=[col for col in columns_to_drop if col in product_catalog_df.columns], errors='ignore')
print(f"[INFO] Shape after dropping unnecessary columns: {product_catalog_df.shape}")

# 3. Handle Null Values
# Remove rows where critical text features ('name', 'main_category', 'sub_category') contain null values.
critical_columns = ['name', 'main_category', 'sub_category']
before_null_drop = len(product_catalog_df)
product_catalog_df = product_catalog_df.dropna(subset=[col for col in critical_columns if col in product_catalog_df.columns])
print(f"[INFO] Removed {before_null_drop - len(product_catalog_df)} rows due to missing critical values. New shape: {product_catalog_df.shape}")

# 4. Handle Duplicate Items
# Remove exact duplicates based on the product name to keep recommendation catalogs unique.
before_dup_drop = len(product_catalog_df)
if 'name' in product_catalog_df.columns:
    product_catalog_df = product_catalog_df.drop_duplicates(subset=['name'], keep='first')
print(f"[INFO] Removed {before_dup_drop - len(product_catalog_df)} duplicate rows. Final cleaned structural shape: {product_catalog_df.shape}")

# 5. Text Cleaning and Preprocessing Pipeline
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Robust text cleaning function
def clean_text_proper(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters, digits, and punctuation using regex (keep alphabets and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize text into individual words
    tokens = word_tokenize(text)
    # Remove stopwords, short tokens (length <= 2), and apply lemmatization
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(cleaned_tokens)

# Combine available text attributes ('name', 'main_category', 'sub_category') to form a rich product document
text_fields = [col for col in ['name', 'main_category', 'sub_category'] if col in product_catalog_df.columns]
product_catalog_df['combined_raw_text'] = product_catalog_df[text_fields].fillna('').agg(' '.join, axis=1)

# Apply text preprocessing pipeline to create our final 'cleaned_text' feature
product_catalog_df['cleaned_text'] = product_catalog_df['combined_raw_text'].apply(clean_text_proper)

# Display sample results to verify data preparation quality
print("\n[SUCCESS] Data Preparation Complete! Sample of processed data:")
display(product_catalog_df[['name', 'main_category', 'sub_category', 'cleaned_text']].head(3))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Please upload your dataset CSV file:


Saving AcmeRetail_Products_70000.csv to AcmeRetail_Products_70000.csv

[INFO] Original dataset shape: (70000, 10)
[INFO] Shape after dropping unnecessary columns: (70000, 7)
[INFO] Removed 0 rows due to missing critical values. New shape: (70000, 7)
[INFO] Removed 6667 duplicate rows. Final cleaned structural shape: (63333, 7)

[SUCCESS] Data Preparation Complete! Sample of processed data:


,name,main_category,sub_category,cleaned_text
0,MyBlush Women Top,women's clothing,Western Wear,myblush woman top womens clothing western wear
1,Ridhi Home & Decor Brass Decorative Weighing S...,appliances,All Appliances,ridhi home decor brass decorative weighing sca...
2,Rangshaily Men's Black White Floral Print Full...,men's clothing,Shirts,rangshaily men black white floral print fullsl...


# Model Building & Model Training

## TF-IDF ( BASELINE MODEL )

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Ensure our cleaned text column exists
if 'cleaned_text' not in product_catalog_df.columns:
    raise ValueError("[ERROR] 'cleaned_text' column not found! Please run the data preparation step first.")

print(f"[INFO] Training TF-IDF Vectorizer on {len(product_catalog_df)} products...")

# 1. Initialize TF-IDF Vectorizer
# We include unigrams and bigrams (ngram_range=(1, 2)) to capture compound terms like "running shoes" or "smart watch".
# We cap max_features at 5000 to maintain efficiency and eliminate rare noise words.
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True # Applies sublinear scaling to term frequency to dampen the weight of overly repeated words
)

# 2. Fit and transform the product catalog's cleaned text corpus
tfidf_matrix = tfidf_vectorizer.fit_transform(product_catalog_df['cleaned_text'])

# 3. Output confirmation of model training
print(f"\n[SUCCESS] TF-IDF Model Training Complete!")
print(f"[INFO] TF-IDF Feature Matrix Shape: {tfidf_matrix.shape}")
print(f"[INFO] Total Unique Vocabulary Size: {len(tfidf_vectorizer.vocabulary_)}")

# Display a sample of the vocabulary features learned
sample_vocab = list(tfidf_vectorizer.vocabulary_.keys())[:10]
print(f"[INFO] Sample vocabulary features: {sample_vocab}")

[INFO] Training TF-IDF Vectorizer on 63333 products...

[SUCCESS] TF-IDF Model Training Complete!
[INFO] TF-IDF Feature Matrix Shape: (63333, 5000)
[INFO] Total Unique Vocabulary Size: 5000
[INFO] Sample vocabulary features: ['woman', 'top', 'womens', 'clothing', 'western', 'wear', 'woman top', 'top womens', 'womens clothing', 'clothing western']


## WORD2VEC

In [4]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 32.5 MB/s eta 0:00:00


In [5]:
from gensim.models import Word2Vec
import numpy as np

# Ensure our cleaned text feature is available in product_catalog_df
if 'cleaned_text' not in product_catalog_df.columns:
    raise ValueError("[ERROR] 'cleaned_text' column not found! Please run the data preparation step first.")

# 1. Prepare tokenized sentences for Word2Vec training
# Gensim Word2Vec expects a list of tokenized sentences (each sentence is a list of string tokens).
tokenized_corpus = [text.split() for text in product_catalog_df['cleaned_text'].dropna() if len(text.split()) > 0]

print(f"[INFO] Training Word2Vec model on {len(tokenized_corpus)} product documents...")

# 2. Initialize and Train Word2Vec Model
# - vector_size=100: Sets the embedding dimensionality to 100 features.
# - window=5: Considers a context window of 5 words before and after the target word.
# - min_count=2: Ignores words that appear fewer than 2 times across the catalog.
# - workers=4: Utilizes multi-threading to speed up training.
# - sg=1: Uses Skip-gram architecture (tends to perform better on rarer words and e-commerce titles).
word2vec_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

print(f"\n[SUCCESS] Word2Vec Model Training Complete!")
print(f"[INFO] Vocabulary Size: {len(word2vec_model.wv.index_to_key)} words")

# 3. Helper Function to generate a product-level vector by averaging its constituent word vectors
def get_product_embedding_w2v(text, model, vector_size=100):
    words = text.split()
    # Filter words present in the trained vocabulary
    valid_words = [word for word in words if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size)
    # Average the word vectors to form a single dense product vector
    return np.mean([model.wv[word] for word in valid_words], axis=0)

# 4. Generate dense vectors for the entire product catalog
product_catalog_df['w2v_vector'] = product_catalog_df['cleaned_text'].apply(
    lambda text: get_product_embedding_w2v(text, word2vec_model)
)

print(f"[INFO] Successfully generated Word2Vec embeddings for all catalog items.")
display(product_catalog_df[['name', 'cleaned_text']].head(2))

[INFO] Training Word2Vec model on 63333 product documents...

[SUCCESS] Word2Vec Model Training Complete!
[INFO] Vocabulary Size: 19918 words
[INFO] Successfully generated Word2Vec embeddings for all catalog items.


,name,cleaned_text
0,MyBlush Women Top,myblush woman top womens clothing western wear
1,Ridhi Home & Decor Brass Decorative Weighing S...,ridhi home decor brass decorative weighing sca...


## FASTTEXT

In [7]:
from gensim.models import FastText
import numpy as np

# Ensure our cleaned text feature is available in product_catalog_df
if 'cleaned_text' not in product_catalog_df.columns:
    raise ValueError("[ERROR] 'cleaned_text' column not found! Please run the data preparation step first.")

# 1. Prepare tokenized sentences for FastText training
tokenized_corpus_ft = [text.split() for text in product_catalog_df['cleaned_text'].dropna() if len(text.split()) > 0]

print(f"[INFO] Training FastText model on {len(tokenized_corpus_ft)} product documents...")

# 2. Initialize and Train FastText Model
# - vector_size=100: Dimensionality of vector embeddings.
# - window=5: Context window size for surrounding words.
# - min_count=2: Ignores words occurring fewer than 2 times.
# - min_n=3, max_n=6: Character n-gram subword bounds (captures roots, prefixes, suffixes).
# - sg=1: Uses Skip-gram architecture.
# - workers=4: Multi-threading for fast execution.
fasttext_model = FastText(
    sentences=tokenized_corpus_ft,
    vector_size=100,
    window=5,
    min_count=2,
    min_n=3,
    max_n=6,
    workers=4,
    sg=1,
    seed=42
)

print(f"\n[SUCCESS] FastText Model Training Complete!")
print(f"[INFO] FastText Vocabulary Size: {len(fasttext_model.wv.index_to_key)} words")

# 3. Helper Function to generate a product-level vector by averaging word embeddings
def get_product_embedding_fasttext(text, model, vector_size=100):
    words = text.split()
    if not words:
        return np.zeros(vector_size)

    word_vectors = []
    for word in words:
        # FastText can generate vectors for ANY word (even unseen/OOV) using subword n-grams!
        word_vectors.append(model.wv[word])

    return np.mean(word_vectors, axis=0)

# 4. Generate dense FastText vectors for the entire product catalog
product_catalog_df['fasttext_vector'] = product_catalog_df['cleaned_text'].apply(
    lambda text: get_product_embedding_fasttext(text, fasttext_model)
)

print(f"[INFO] Successfully generated FastText embeddings for all catalog items.")
display(product_catalog_df[['name', 'cleaned_text']].head(2))

[INFO] Training FastText model on 63333 product documents...

[SUCCESS] FastText Model Training Complete!
[INFO] FastText Vocabulary Size: 19918 words
[INFO] Successfully generated FastText embeddings for all catalog items.


,name,cleaned_text
0,MyBlush Women Top,myblush woman top womens clothing western wear
1,Ridhi Home & Decor Brass Decorative Weighing S...,ridhi home decor brass decorative weighing sca...


# Model Testing

## TF-IDF TESTING

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend_products_tfidf(query_input, product_catalog_df, tfidf_vectorizer, tfidf_matrix, top_n=5):
    """
    Performs similarity search and ranking using the TF-IDF model and cosine similarity.
    Accepts either an integer index or a raw query string, ensuring identical preprocessing.
    """
    # 1. Handle query input type (Index vs Plain Text)
    if isinstance(query_input, int):
        if query_input < 0 or query_input >= len(product_catalog_df):
            raise IndexError(f"[ERROR] Index {query_input} is out of bounds for catalog of size {len(product_catalog_df)}.")
        query_vector = tfidf_matrix[query_input]
        query_display_name = product_catalog_df.iloc[query_input]['name']
        exclude_self_idx = query_input
        print(f"[INFO] Query Product (Index {query_input}): '{query_display_name}'")

    elif isinstance(query_input, str):
        # FIX APPLIED: Apply the exact same text cleaning pipeline used during training!
        cleaned_query = clean_text_proper(query_input)
        if not cleaned_query:
            raise ValueError(f"[ERROR] Query '{query_input}' resulted in empty text after preprocessing (e.g., contained only stop words).")

        query_vector = tfidf_vectorizer.transform([cleaned_query])
        exclude_self_idx = None
        print(f"[INFO] Raw Query: '{query_input}'")
        print(f"[INFO] Preprocessed Query: '{cleaned_query}'")

    else:
        raise TypeError("[ERROR] query_input must be an integer index or a string.")

    # 2. Calculate Cosine Similarity across the entire TF-IDF matrix
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # 3. Sort similarity scores in descending order (highest relevance first)
    ranked_indices = similarity_scores.argsort()[::-1]

    # 4. Collect Top-N Recommendations (filtering out self-match if indexed query)
    recommendations = []
    for idx in ranked_indices:
        if exclude_self_idx is not None and idx == exclude_self_idx:
            continue # Skip the query product itself
        if len(recommendations) >= top_n:
            break

        recommendations.append({
            'Catalog Index': idx,
            'Product Name': product_catalog_df.iloc[idx]['name'],
            'Main Category': product_catalog_df.iloc[idx].get('main_category', 'N/A'),
            'Sub Category': product_catalog_df.iloc[idx].get('sub_category', 'N/A'),
            'Similarity Score (TF-IDF)': round(float(similarity_scores[idx]), 4)
        })

    # Convert results into a clean presentation DataFrame
    recommendations_df = pd.DataFrame(recommendations)
    return recommendations_df

# --- Test the TF-IDF Similarity Search (Testing both Index and Text Queries) ---

print("--- Test 1: Query by Catalog Index ---")
display(recommend_products_tfidf(query_input=0, product_catalog_df=product_catalog_df, tfidf_vectorizer=tfidf_vectorizer, tfidf_matrix=tfidf_matrix, top_n=5))

print("\n--- Test 2: Query by Raw Plain Text ---")
display(recommend_products_tfidf(query_input="Wireless Gaming Mouse with RGB", product_catalog_df=product_catalog_df, tfidf_vectorizer=tfidf_vectorizer, tfidf_matrix=tfidf_matrix, top_n=5))

--- Test 1: Query by Catalog Index ---
[INFO] Query Product (Index 0): 'MyBlush Women Top'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (TF-IDF)
0,5223,Selvia Women Top,women's clothing,Western Wear,1.0
1,11131,Lyriq Women Tops,women's clothing,Western Wear,1.0
2,53262,MISVERSE Women Top,women's clothing,Western Wear,1.0
3,9680,SIDDHANAM Women Top,women's clothing,Western Wear,1.0
4,60067,Selvia Women TOP,women's clothing,Western Wear,1.0



--- Test 2: Query by Raw Plain Text ---
[INFO] Raw Query: 'Wireless Gaming Mouse with RGB'
[INFO] Preprocessed Query: 'wireless gaming mouse rgb'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (TF-IDF)
0,57731,Ant Esports USB GM600 RGB Wired Programmable G...,"tv, audio & cameras",All Electronics,0.6533
1,39769,"Redgear A-17 Gaming Mouse with Upto 6400 DPI, ...","tv, audio & cameras",All Electronics,0.6365
2,38697,Razer Basilisk V3 Customizable Ergonomic Gamin...,"tv, audio & cameras",All Electronics,0.5514
3,17572,"Tukzer RGB Gaming Mouse Pad with 4 USB HUB, La...","tv, audio & cameras",All Electronics,0.5223
4,13800,Logitech G502 Hero High Performance Wired Gami...,"tv, audio & cameras",All Electronics,0.4988


## WORD2VEC TESTING

In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def recommend_products_word2vec(query_input, product_catalog_df, word2vec_model, top_n=5):
    """
    Performs similarity search and ranking using the trained Word2Vec model and cosine similarity.
    Accepts either an integer index or a raw query string.
    """
    # 1. Handle query input type (Index vs Plain Text)
    if isinstance(query_input, int):
        if query_input < 0 or query_input >= len(product_catalog_df):
            raise IndexError(f"[ERROR] Index {query_input} is out of bounds for catalog of size {len(product_catalog_df)}.")
        query_vector = product_catalog_df.iloc[query_input]['w2v_vector']
        query_display_name = product_catalog_df.iloc[query_input]['name']
        exclude_self_idx = query_input
        print(f"[INFO] Query Product (Index {query_input}): '{query_display_name}'")

    elif isinstance(query_input, str):
        # Apply identical text cleaning pipeline used during training
        cleaned_query = clean_text_proper(query_input)
        if not cleaned_query:
            raise ValueError(f"[ERROR] Query '{query_input}' resulted in empty text after preprocessing.")

        # Generate embedding for the query using our Word2Vec helper function
        query_vector = get_product_embedding_w2v(cleaned_query, word2vec_model)
        exclude_self_idx = None
        print(f"[INFO] Raw Query: '{query_input}'")
        print(f"[INFO] Preprocessed Query: '{cleaned_query}'")

    else:
        raise TypeError("[ERROR] query_input must be an integer index or a string.")

    # Ensure query vector is shaped correctly (2D array for cosine_similarity)
    query_vector = np.array(query_vector).reshape(1, -1)

    # Stack all catalog word2vec vectors into a matrix array
    catalog_vectors = np.vstack(product_catalog_df['w2v_vector'].values)

    # 2. Calculate Cosine Similarity across all catalog word2vec embeddings
    similarity_scores = cosine_similarity(query_vector, catalog_vectors).flatten()

    # 3. Sort similarity scores in descending order (highest relevance first)
    ranked_indices = similarity_scores.argsort()[::-1]

    # 4. Collect Top-N Recommendations (filtering out self-match if indexed query)
    recommendations = []
    for idx in ranked_indices:
        if exclude_self_idx is not None and idx == exclude_self_idx:
            continue # Skip the query product itself
        if len(recommendations) >= top_n:
            break

        recommendations.append({
            'Catalog Index': idx,
            'Product Name': product_catalog_df.iloc[idx]['name'],
            'Main Category': product_catalog_df.iloc[idx].get('main_category', 'N/A'),
            'Sub Category': product_catalog_df.iloc[idx].get('sub_category', 'N/A'),
            'Similarity Score (Word2Vec)': round(float(similarity_scores[idx]), 4)
        })

    # Convert results into a clean presentation DataFrame
    recommendations_df = pd.DataFrame(recommendations)
    return recommendations_df

# --- Test the Word2Vec Similarity Search ---
print("--- Test 1: Query by Catalog Index (Word2Vec) ---")
display(recommend_products_word2vec(query_input=0, product_catalog_df=product_catalog_df, word2vec_model=word2vec_model, top_n=5))

print("\n--- Test 2: Query by Raw Plain Text (Word2Vec) ---")
display(recommend_products_word2vec(query_input="Wireless Gaming Mouse with RGB", product_catalog_df=product_catalog_df, word2vec_model=word2vec_model, top_n=5))

--- Test 1: Query by Catalog Index (Word2Vec) ---
[INFO] Query Product (Index 0): 'MyBlush Women Top'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (Word2Vec)
0,53262,MISVERSE Women Top,women's clothing,Western Wear,0.9999
1,9680,SIDDHANAM Women Top,women's clothing,Western Wear,0.9998
2,11131,Lyriq Women Tops,women's clothing,Western Wear,0.9996
3,60067,Selvia Women TOP,women's clothing,Western Wear,0.9992
4,5223,Selvia Women Top,women's clothing,Western Wear,0.9992



--- Test 2: Query by Raw Plain Text (Word2Vec) ---
[INFO] Raw Query: 'Wireless Gaming Mouse with RGB'
[INFO] Preprocessed Query: 'wireless gaming mouse rgb'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (Word2Vec)
0,31451,Belear BL-F40 TWS Gaming Wireless Bluetooth Ea...,"tv, audio & cameras",Headphones,0.9173
1,28818,Portronics Dash 12 60W Wireless Bluetooth Part...,"tv, audio & cameras",Speakers,0.9171
2,5471,Blaupunkt Newly Launched SBA15 Gaming 15W Blue...,"tv, audio & cameras",Speakers,0.9170
3,47726,LANDMARK LM BH136 True Wireless Gaming Earbuds...,"tv, audio & cameras",Cameras,0.9166
4,23922,Honeywell Moxie V100 3W Wireless Bluetooth V5....,"tv, audio & cameras",Speakers,0.9152


## FastText testing

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def recommend_products_fasttext(query_input, product_catalog_df, fasttext_model, top_n=5):
    """
    Performs similarity search and ranking using the trained FastText model and cosine similarity.
    Accepts either an integer index or a raw query string, utilizing subword features for OOV handling.
    """
    # 1. Handle query input type (Index vs Plain Text)
    if isinstance(query_input, int):
        if query_input < 0 or query_input >= len(product_catalog_df):
            raise IndexError(f"[ERROR] Index {query_input} is out of bounds for catalog size {len(product_catalog_df)}.")
        query_vector = product_catalog_df.iloc[query_input]['fasttext_vector']
        query_display_name = product_catalog_df.iloc[query_input]['name']
        exclude_self_idx = query_input
        print(f"[INFO] Query Product (Index {query_input}): '{query_display_name}'")

    elif isinstance(query_input, str):
        # Apply identical text cleaning pipeline used during training
        cleaned_query = clean_text_proper(query_input)
        if not cleaned_query:
            raise ValueError(f"[ERROR] Query '{query_input}' resulted in empty text after preprocessing.")

        # Generate embedding for the query using FastText helper function (handles OOV/typos via subwords)
        query_vector = get_product_embedding_fasttext(cleaned_query, fasttext_model)
        exclude_self_idx = None
        print(f"[INFO] Raw Query: '{query_input}'")
        print(f"[INFO] Preprocessed Query: '{cleaned_query}'")

    else:
        raise TypeError("[ERROR] query_input must be an integer index or a string.")

    # Ensure query vector is shaped correctly (2D array for cosine_similarity)
    query_vector = np.array(query_vector).reshape(1, -1)

    # Stack all catalog fasttext vectors into a matrix array
    catalog_vectors = np.vstack(product_catalog_df['fasttext_vector'].values)

    # 2. Calculate Cosine Similarity across all catalog fasttext embeddings
    similarity_scores = cosine_similarity(query_vector, catalog_vectors).flatten()

    # 3. Sort similarity scores in descending order (highest relevance first)
    ranked_indices = similarity_scores.argsort()[::-1]

    # 4. Collect Top-N Recommendations (filtering out self-match if indexed query)
    recommendations = []
    for idx in ranked_indices:
        if exclude_self_idx is not None and idx == exclude_self_idx:
            continue # Skip the query product itself
        if len(recommendations) >= top_n:
            break

        recommendations.append({
            'Catalog Index': idx,
            'Product Name': product_catalog_df.iloc[idx]['name'],
            'Main Category': product_catalog_df.iloc[idx].get('main_category', 'N/A'),
            'Sub Category': product_catalog_df.iloc[idx].get('sub_category', 'N/A'),
            'Similarity Score (FastText)': round(float(similarity_scores[idx]), 4)
        })

    # Convert results into a clean presentation DataFrame
    recommendations_df = pd.DataFrame(recommendations)
    return recommendations_df

# --- Test the FastText Similarity Search ---
print("--- Test 1: Query by Catalog Index (FastText) ---")
display(recommend_products_fasttext(query_input=0, product_catalog_df=product_catalog_df, fasttext_model=fasttext_model, top_n=5))

print("\n--- Test 2: Query with Typo / Unseen Term (FastText Isolated Test) ---")
display(recommend_products_fasttext(query_input="bluetoth wireles earphone", product_catalog_df=product_catalog_df, fasttext_model=fasttext_model, top_n=5))

--- Test 1: Query by Catalog Index (FastText) ---
[INFO] Query Product (Index 0): 'MyBlush Women Top'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (FastText)
0,11131,Lyriq Women Tops,women's clothing,Western Wear,0.9989
1,53262,MISVERSE Women Top,women's clothing,Western Wear,0.9987
2,5223,Selvia Women Top,women's clothing,Western Wear,0.9984
3,60067,Selvia Women TOP,women's clothing,Western Wear,0.9984
4,56309,Women Ponchos Top,women's clothing,Western Wear,0.9981



--- Test 2: Query with Typo / Unseen Term (FastText Isolated Test) ---
[INFO] Raw Query: 'bluetoth wireles earphone'
[INFO] Preprocessed Query: 'bluetoth wireles earphone'


,Catalog Index,Product Name,Main Category,Sub Category,Similarity Score (FastText)
0,23175,Luisport Open Ear Bluetooth Headphones Wireles...,"tv, audio & cameras",Headphones,0.9813
1,3359,Wireless Bluetooth Headphones Earphone For viv...,"tv, audio & cameras",Headphones,0.9763
2,55080,Siwi Wireless Bluetooth Earphones for Xiaomi M...,"tv, audio & cameras",Headphones,0.9756
3,26370,Wireless Bluetooth Headphones Earphone For One...,"tv, audio & cameras",Headphones,0.9754
4,41651,Wireless Bluetooth Headphones Earphones for Xi...,"tv, audio & cameras",Headphones,0.9747


## Hybrid Word2Vec + FastText

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def recommend_products_hybrid(query_input, product_catalog_df, word2vec_model, fasttext_model, top_n=5):
    """
    Performs hybrid similarity search using Word2Vec as the primary model and
    FastText as an automatic fallback when Out-Of-Vocabulary (OOV) words or typos are detected.
    """
    # 1. Handle query input type (Index vs Plain Text)
    if isinstance(query_input, int):
        if query_input < 0 or query_input >= len(product_catalog_df):
            raise IndexError(f"[ERROR] Index {query_input} is out of bounds for catalog size {len(product_catalog_df)}.")
        # Index queries always use primary Word2Vec precomputed vectors
        query_vector = product_catalog_df.iloc[query_input]['w2v_vector']
        query_display_name = product_catalog_df.iloc[query_input]['name']
        exclude_self_idx = query_input
        model_used = "Word2Vec (Catalog Index)"
        print(f"[INFO] Query Product (Index {query_input}): '{query_display_name}'")

    elif isinstance(query_input, str):
        # Clean plain text query
        cleaned_query = clean_text_proper(query_input)
        if not cleaned_query:
            raise ValueError(f"[ERROR] Query '{query_input}' resulted in empty text after preprocessing.")

        words = cleaned_query.split()

        # Check vocabulary coverage for Word2Vec
        vocab_w2v = word2vec_model.wv
        unknown_words = [word for word in words if word not in vocab_w2v]
        oov_ratio = len(unknown_words) / len(words) if len(words) > 0 else 0.0

        # Fallback Logic: If OOV ratio > 0 (contains unseen words/typos), trigger FastText fallback!
        exclude_self_idx = None
        if oov_ratio > 0.0:
            model_used = f"FastText (Fallback Triggered: {len(unknown_words)} OOV word(s) detected -> {unknown_words})"
            query_vector = get_product_embedding_fasttext(cleaned_query, fasttext_model)
            # Use FastText catalog vectors for comparison
            catalog_vectors = np.vstack(product_catalog_df['fasttext_vector'].values)
        else:
            model_used = "Word2Vec (Primary Model: Full Vocabulary Match)"
            query_vector = get_product_embedding_w2v(cleaned_query, word2vec_model)
            # Use Word2Vec catalog vectors for comparison
            catalog_vectors = np.vstack(product_catalog_df['w2v_vector'].values)

        print(f"[INFO] Raw Query: '{query_input}'")
        print(f"[INFO] Preprocessed Query: '{cleaned_query}'")
        print(f"[INFO] Routing Decision Engine $\rightarrow$ Selected Model: {model_used}")

    else:
        raise TypeError("[ERROR] query_input must be an integer index or a string.")

    # If query_input was an index, default catalog vectors to Word2Vec space
    if isinstance(query_input, int):
        catalog_vectors = np.vstack(product_catalog_df['w2v_vector'].values)

    # 2. Ensure query vector is shaped correctly (2D array for cosine_similarity)
    query_vector = np.array(query_vector).reshape(1, -1)

    # 3. Calculate Cosine Similarity
    similarity_scores = cosine_similarity(query_vector, catalog_vectors).flatten()

    # 4. Sort similarity scores in descending order (highest relevance first)
    ranked_indices = similarity_scores.argsort()[::-1]

    # 5. Collect Top-N Recommendations (filtering out self-match if indexed query)
    recommendations = []
    for idx in ranked_indices:
        if exclude_self_idx is not None and idx == exclude_self_idx:
            continue
        if len(recommendations) >= top_n:
            break

        recommendations.append({
            'Catalog Index': idx,
            'Product Name': product_catalog_df.iloc[idx]['name'],
            'Main Category': product_catalog_df.iloc[idx].get('main_category', 'N/A'),
            'Sub Category': product_catalog_df.iloc[idx].get('sub_category', 'N/A'),
            'Model Route': model_used.split(' ')[0],
            'Similarity Score': round(float(similarity_scores[idx]), 4)
        })

    # Convert results into a clean presentation DataFrame
    recommendations_df = pd.DataFrame(recommendations)
    return recommendations_df

# --- Test the Hybrid Model ---
print("--- Test 1: Standard Query (Primary Word2Vec Route) ---")
display(recommend_products_hybrid("wireless mouse", product_catalog_df, word2vec_model, fasttext_model, top_n=5))

print("\n--- Test 2: Query with Typo / Unseen Term (FastText Fallback Route) ---")
display(recommend_products_hybrid("bluetoth wireles earphone", product_catalog_df, word2vec_model, fasttext_model, top_n=5))

--- Test 1: Standard Query (Primary Word2Vec Route) ---
[INFO] Raw Query: 'wireless mouse'
[INFO] Preprocessed Query: 'wireless mouse'
ightarrow$ Selected Model: Word2Vec (Primary Model: Full Vocabulary Match)


,Catalog Index,Product Name,Main Category,Sub Category,Model Route,Similarity Score
0,59823,Glaceon Mini Bluetooth Animal Wireless Speaker...,"tv, audio & cameras",Speakers,Word2Vec,0.9061
1,42545,JNUOBI Bone Conduction Motion Bluetooth Wirele...,"tv, audio & cameras",Headphones,Word2Vec,0.8753
2,3820,Corseca Just Sushi Grande Wireless Speaker40W ...,"tv, audio & cameras",Speakers,Word2Vec,0.8703
3,56922,Urban Audio Mini 2 Wireless Bluetooth 5.0 Port...,"tv, audio & cameras",Speakers,Word2Vec,0.8610
4,41959,M19 TWS Bluetooth 5.0 Wireless Earbuds Touch W...,"tv, audio & cameras",Headphones,Word2Vec,0.8604



--- Test 2: Query with Typo / Unseen Term (FastText Fallback Route) ---
[INFO] Raw Query: 'bluetoth wireles earphone'
[INFO] Preprocessed Query: 'bluetoth wireles earphone'
ightarrow$ Selected Model: FastText (Fallback Triggered: 1 OOV word(s) detected -> ['bluetoth'])


,Catalog Index,Product Name,Main Category,Sub Category,Model Route,Similarity Score
0,23175,Luisport Open Ear Bluetooth Headphones Wireles...,"tv, audio & cameras",Headphones,FastText,0.9813
1,3359,Wireless Bluetooth Headphones Earphone For viv...,"tv, audio & cameras",Headphones,FastText,0.9763
2,55080,Siwi Wireless Bluetooth Earphones for Xiaomi M...,"tv, audio & cameras",Headphones,FastText,0.9756
3,26370,Wireless Bluetooth Headphones Earphone For One...,"tv, audio & cameras",Headphones,FastText,0.9754
4,41651,Wireless Bluetooth Headphones Earphones for Xi...,"tv, audio & cameras",Headphones,FastText,0.9747


# Model Evaluation

In [14]:
import pandas as pd

def evaluate_models_side_by_side(query_input, product_catalog_df, tfidf_vectorizer, tfidf_matrix, word2vec_model, fasttext_model, top_n=5):
    """
    Executes similarity search across TF-IDF, Word2Vec, and FastText models for a given query,
    and returns a unified side-by-side comparison table showing rankings and similarity scores.
    """
    print(f"[INFO] Running multi-model evaluation for query input: '{query_input}'\n")

    # 1. Get recommendations from each model using our previously defined functions
    tfidf_results = recommend_products_tfidf(query_input, product_catalog_df, tfidf_vectorizer, tfidf_matrix, top_n)
    w2v_results = recommend_products_word2vec(query_input, product_catalog_df, word2vec_model, top_n)
    ft_results = recommend_products_fasttext(query_input, product_catalog_df, fasttext_model, top_n)

    # 2. Build a unified side-by-side comparison dataframe
    comparison_data = []
    for rank in range(top_n):
        comparison_data.append({
            'Rank': rank + 1,
            'TF-IDF Recommended Product': tfidf_results.iloc[rank]['Product Name'] if rank < len(tfidf_results) else 'N/A',
            'TF-IDF Score': tfidf_results.iloc[rank].get('Similarity Score (TF-IDF)', tfidf_results.iloc[rank].get('Similarity Score', 0.0)) if rank < len(tfidf_results) else 0.0,
            'Word2Vec Recommended Product': w2v_results.iloc[rank]['Product Name'] if rank < len(w2v_results) else 'N/A',
            'Word2Vec Score': w2v_results.iloc[rank].get('Similarity Score (Word2Vec)', w2v_results.iloc[rank].get('Similarity Score', 0.0)) if rank < len(w2v_results) else 0.0,
            'FastText Recommended Product': ft_results.iloc[rank]['Product Name'] if rank < len(ft_results) else 'N/A',
            'FastText Score': ft_results.iloc[rank].get('Similarity Score (FastText)', ft_results.iloc[rank].get('Similarity Score', 0.0)) if rank < len(ft_results) else 0.0,
        })

    comparison_df = pd.DataFrame(comparison_data)
    return comparison_df

# --- Execute Side-by-Side Model Evaluation ---
# Test using a descriptive plain-text search query
evaluation_query = "tops"
side_by_side_table = evaluate_models_side_by_side(
    query_input=evaluation_query,
    product_catalog_df=product_catalog_df,
    tfidf_vectorizer=tfidf_vectorizer,
    tfidf_matrix=tfidf_matrix,
    word2vec_model=word2vec_model,
    fasttext_model=fasttext_model,
    top_n=5
)

print(f"[SUCCESS] Model Evaluation Comparison Table Generated:")
display(side_by_side_table)

[INFO] Running multi-model evaluation for query input: 'tops'

[INFO] Raw Query: 'tops'
[INFO] Preprocessed Query: 'top'
[INFO] Raw Query: 'tops'
[INFO] Preprocessed Query: 'top'
[INFO] Raw Query: 'tops'
[INFO] Preprocessed Query: 'top'
[SUCCESS] Model Evaluation Comparison Table Generated:


,Rank,TF-IDF Recommended Product,TF-IDF Score,Word2Vec Recommended Product,Word2Vec Score,FastText Recommended Product,FastText Score
0,1,TOPLOT Crop Top for Women||Tops for Women||Cro...,0.4904,Lymio Tops for Women || Tops for Women Women T...,0.8550,Lymio Tops for Women || Tops for Women Women T...,0.8686
1,2,Popuppataka Girl's Cotton Top Shirt wear-White...,0.4724,Istyle Can Women's & Girls' Solid Ribbed Squar...,0.8291,Istyle Can Women's & Girls' Solid Ribbed Squar...,0.8248
2,3,Leriya Fashion Womens Tops Casual Rayon Blouse...,0.4537,Style n feel®Women's & Girls' Solid Ribbed Cro...,0.8025,NiloBerry Tops for Women | Crop top for Women ...,0.8211
3,4,J B Fashion Women Top with Full Sleeves for Wo...,0.4508,BLINKIN Women's Crop Tops | Ribbed Crop Tank T...,0.7994,Style n feel®Women's & Girls' Solid Ribbed Cro...,0.8183
4,5,J B Fashion Women Top with Half Sleeves for Wo...,0.4497,NiloBerry Tops for Women | Crop top for Women ...,0.7964,BLINKIN Women's Crop Tops | Ribbed Crop Tank T...,0.8168


# Model Deployment

In [15]:
import joblib
import pickle
from google.colab import files

print("[INFO] Starting model serialization and artifact export...")

# 1. Define filenames for our saved artifacts
catalog_filename = "product_catalog_vectorized.pkl"
tfidf_model_filename = "tfidf_vectorizer.pkl"
w2v_model_filename = "word2vec_model.model"
ft_model_filename = "fasttext_model.model"

# 2. Serialize and export the product catalog DataFrame (including precomputed vectors) using joblib/pickle
# joblib is highly optimized for numpy-heavy pandas DataFrames.
joblib.dump(product_catalog_df, catalog_filename)
print(f"[SUCCESS] Exported: {catalog_filename}")

# 3. Serialize and export the TF-IDF Vectorizer and Matrix artifacts
joblib.dump(tfidf_vectorizer, tfidf_model_filename)
print(f"[SUCCESS] Exported: {tfidf_model_filename}")

# 4. Export Gensim Word2Vec and FastText models using their native .save() methods
# Gensim models have specialized internal structures, so native saving preserves vocabulary and weights best.
word2vec_model.save(w2v_model_filename)
print(f"[SUCCESS] Exported: {w2v_model_filename}")

fasttext_model.save(ft_model_filename)
print(f"[SUCCESS] Exported: {ft_model_filename}")

print("\n[INFO] All artifacts successfully packaged! You can download them directly from your Colab file explorer.")

# Optional: Automatically trigger downloads for local backup in Colab
files.download(catalog_filename)
files.download(tfidf_model_filename)
files.download(w2v_model_filename)
files.download(ft_model_filename)

[INFO] Starting model serialization and artifact export...
[SUCCESS] Exported: product_catalog_vectorized.pkl
[SUCCESS] Exported: tfidf_vectorizer.pkl
[SUCCESS] Exported: word2vec_model.model
[SUCCESS] Exported: fasttext_model.model

[INFO] All artifacts successfully packaged! You can download them directly from your Colab file explorer.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import joblib
import pandas as pd
import numpy as np
from gensim.models import Word2Vec, FastText
from sklearn.metrics.pairwise import cosine_similarity

print("[INFO] Starting verification test for saved deployment artifacts...\n")

# 1. Load serialized artifacts from disk
loaded_catalog_df = joblib.load("product_catalog_vectorized.pkl")
loaded_tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
loaded_word2vec_model = Word2Vec.load("word2vec_model.model")
loaded_fasttext_model = FastText.load("fasttext_model.model")

print(f"[SUCCESS] Successfully loaded product catalog! Total rows: {len(loaded_catalog_df)}")
print(f"[SUCCESS] Successfully loaded TF-IDF Vectorizer! Vocabulary size: {len(loaded_tfidf_vectorizer.vocabulary_)}")
print(f"[SUCCESS] Successfully loaded Word2Vec Model! Vocabulary size: {len(loaded_word2vec_model.wv.index_to_key)}")
print(f"[SUCCESS] Successfully loaded FastText Model! Vocabulary size: {len(loaded_fasttext_model.wv.index_to_key)}")

# 2. Quick Smoke Test: Run a test search query using the loaded FastText model
test_query = "wireless mouse"
cleaned_test_query = clean_text_proper(test_query)
query_vector = np.array(get_product_embedding_fasttext(cleaned_test_query, loaded_fasttext_model)).reshape(1, -1)
catalog_vectors = np.vstack(loaded_catalog_df['fasttext_vector'].values)

# Compute similarity scores
similarity_scores = cosine_similarity(query_vector, catalog_vectors).flatten()
top_idx = similarity_scores.argmax()

print(f"\n[SMOKE TEST RESULT]")
print(f"Query: '{test_query}'")
print(f"Top Recommended Product: '{loaded_catalog_df.iloc[top_idx]['name']}'")
print(f"Similarity Score: {round(float(similarity_scores[top_idx]), 4)}")
print("\n[ALL TESTS PASSED] Saved models are fully functional and production-ready!")

[INFO] Starting verification test for saved deployment artifacts...

[SUCCESS] Successfully loaded product catalog! Total rows: 63333
[SUCCESS] Successfully loaded TF-IDF Vectorizer! Vocabulary size: 5000
[SUCCESS] Successfully loaded Word2Vec Model! Vocabulary size: 19918
[SUCCESS] Successfully loaded FastText Model! Vocabulary size: 19918

[SMOKE TEST RESULT]
Query: 'wireless mouse'
Top Recommended Product: 'Glaceon Mini Bluetooth Animal Wireless Speaker,Stereo Tiny Cute Wireless Speaker,Stereo Pairing Speaker with Wireless Char...'
Similarity Score: 0.898

[ALL TESTS PASSED] Saved models are fully functional and production-ready!
